In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

rng = np.random.default_rng(20260820)

n_units = 2400

unit_ids = [f"WHRM-{i:05d}" for i in range(1, n_units + 1)]
build_dates = pd.date_range("2026-07-01", "2026-07-28", periods=n_units)

production_lines = rng.choice(
    ["Line A", "Line B", "Line C"],
    size=n_units,
    p=[0.42, 0.33, 0.25]
)

shifts = rng.choice(
    ["Day", "Evening"],
    size=n_units,
    p=[0.62, 0.38]
)

test_stations = rng.choice(
    ["TS-01", "TS-02", "TS-03"],
    size=n_units,
    p=[0.40, 0.35, 0.25]
)

firmware_versions = np.where(
    build_dates < pd.Timestamp("2026-07-15"),
    "FW-SIM-0.2",
    "FW-SIM-0.3"
)

heart_rate_error_bpm = rng.normal(loc=2.4, scale=1.35, size=n_units)
response_time_seconds = rng.normal(loc=3.85, scale=0.62, size=n_units)
signal_quality_score = rng.normal(loc=91.5, scale=5.2, size=n_units)

line_b_mask = production_lines == "Line B"
test_station_03_mask = test_stations == "TS-03"
old_firmware_mask = firmware_versions == "FW-SIM-0.2"

heart_rate_error_bpm[line_b_mask] += 0.55
response_time_seconds[old_firmware_mask] += 0.50
signal_quality_score[test_station_03_mask] -= 3.5

heart_rate_error_bpm = np.clip(heart_rate_error_bpm, 0.1, None)
response_time_seconds = np.clip(response_time_seconds, 1.5, None)
signal_quality_score = np.clip(signal_quality_score, 50, 100)

defect_category = np.full(n_units, "None", dtype=object)

low_signal_quality = signal_quality_score < 82
slow_response = response_time_seconds > 5.0
high_hr_error = heart_rate_error_bpm > 5.0

defect_category[low_signal_quality] = "Low signal quality"
defect_category[slow_response] = "Slow response time"
defect_category[high_hr_error] = "High heart-rate error"

assembly_defect_mask = rng.random(n_units) < 0.018
cosmetic_defect_mask = rng.random(n_units) < 0.012

defect_category[assembly_defect_mask] = "Assembly defect"
defect_category[cosmetic_defect_mask] = "Cosmetic defect"

test_status = np.where(defect_category == "None", "Pass", "Fail")

rework_required = np.where(
    test_status == "Fail",
    rng.choice(["Yes", "No"], size=n_units, p=[0.72, 0.28]),
    "No"
)

df = pd.DataFrame({
    "unit_id": unit_ids,
    "build_date": build_dates.strftime("%Y-%m-%d"),
    "production_line": production_lines,
    "shift": shifts,
    "test_station": test_stations,
    "firmware_version": firmware_versions,
    "heart_rate_error_bpm": np.round(heart_rate_error_bpm, 2),
    "response_time_seconds": np.round(response_time_seconds, 2),
    "signal_quality_score": np.round(signal_quality_score, 1),
    "test_status": test_status,
    "defect_category": defect_category,
    "rework_required": rework_required
})

df.to_csv("synthetic_manufacturing_data.csv", index=False)

print(f"Created {len(df):,} synthetic manufacturing records.")
print(f"Overall first-pass yield: {(df['test_status'] == 'Pass').mean():.1%}")
display(df.head())
display(df["defect_category"].value_counts())

Created 2,400 synthetic manufacturing records.
Overall first-pass yield: 79.4%


,unit_id,build_date,production_line,shift,test_station,firmware_version,heart_rate_error_bpm,response_time_seconds,signal_quality_score,test_status,defect_category,rework_required
0,WHRM-00001,2026-07-01,Line A,Day,TS-01,FW-SIM-0.2,0.76,3.81,95.0,Pass,None,No
1,WHRM-00002,2026-07-01,Line A,Evening,TS-01,FW-SIM-0.2,4.25,3.77,92.1,Pass,None,No
2,WHRM-00003,2026-07-01,Line A,Evening,TS-02,FW-SIM-0.2,3.42,5.00,94.4,Pass,None,No
3,WHRM-00004,2026-07-01,Line B,Day,TS-03,FW-SIM-0.2,1.93,4.11,94.2,Pass,None,No
4,WHRM-00005,2026-07-01,Line A,Evening,TS-01,FW-SIM-0.2,0.91,5.18,89.0,Fail,Slow response time,Yes


,count
defect_category,
None,1906
Slow response time,205
Low signal quality,114
High heart-rate error,100
Assembly defect,47
Cosmetic defect,28
